In [1]:
# -*- encoding: utf-8 -*-


In [2]:
from dataset.cell_motion import CellMotion
from model.simulator import Simulator
import torch
from torch_geometric.loader import DataLoader
import torch_geometric.transforms as T
import os
import csv
from datetime import datetime
from typing import Optional

# 配置参数
# ============================================

# 数据集路径
dataset_dir = os.path.join(os.getcwd(), 'dataset')

# 训练参数
batch_size = 32  # 批次大小
sequence_length = 20  # 输入的历史时间步数
max_epochs = 1500  # 最大训练轮数

# 网络参数
message_passing_num = 15  # 消息传递层数
node_input_size = sequence_length * 2 + 1  # 节点输入维度 = 输入的历史时间步数 * 2(x,y坐标) + 1(时间)
edge_input_size = 3  # 边输入维度（笛卡尔坐标差 + 距离）
learning_rate = 1e-4  # 学习率

# 训练控制
print_batch = 10  # 每10个批次打印一次损失
save_epoch = 5  # 每200个批次保存一次模型检查点

In [4]:
# 设置计算设备：如果CUDA可用则使用GPU，否则使用CPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# 初始化模拟器模型
# 输入：历史位置序列 + 时间
# 输出：下一时刻的位置（2D坐标）

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
save_dir=os.path.join(f'checkpoint',timestamp)
os.mkdir(save_dir)
simulator = Simulator(
    message_passing_num=message_passing_num,
    node_input_size=node_input_size,
    edge_input_size=edge_input_size,
    device=device,
    model_dir=save_dir  # 使用不同的检查点文件
)

# 使用Adam优化器
optimizer = torch.optim.Adam(simulator.parameters(), lr=learning_rate)
print('Optimizer initialized')

Using device: cuda
Simulator model initialized
Optimizer initialized


In [5]:
def train(model: Simulator, dataloader, optimizer, current_epoch, *,
          max_grad_norm: float = 1.0,
          max_grad_value: Optional[float] = None):
    """
    训练函数（包含梯度裁剪）

    参数:
        model: Simulator模型实例（注意：训练中 optimizer 应包含要优化的参数）
        dataloader: 数据加载器
        optimizer: 优化器
        current_epoch: 当前epoch编号
        max_grad_norm: 如果为正，则在 optimizer.step() 前应用按范数裁剪，常用值例如 1.0
        max_grad_value: 如果不为 None，则在按范数裁剪后对梯度按值裁剪到 [-max_grad_value, max_grad_value]
    """
    epoch_loss = 0.0
    batch_count = 0

    # 确保处于训练模式
    try:
        model.train()
    except Exception:
        pass
        
    for batch_index, graph in enumerate(dataloader):
        graph = transformer(graph)
        # 将图数据移到GPU（外部需定义 device）
        graph = graph.to(device)
        # 提取节点特征和目标
        node_features = graph.x  # (N, node_input_size)
        target_positions = graph.y  # (N, 2) - 下一时刻的位置
        current_positions = graph.pos  # (N, 2) - 当前位置
        # 前向传播，预测位置变化（displacement）
        predicted_output = model.model(graph)  # (N, 2)

        # 计算位置变化（displacement）
        target_displacement = target_positions - current_positions

        # 归一化目标
        target_displacement_normalized = model._output_normalizer(
            target_displacement,
            model.training
        )
        # 计算均方误差损失
        mse_loss = torch.mean((predicted_output - target_displacement_normalized) ** 2)
        
        # 反归一化预测的位移
        predicted_displacement = model._output_normalizer.inverse(predicted_output)
        
        # 计算当前时刻质心
        current_centroid = torch.mean(current_positions, dim=0)  # (2,)
        
        # 计算预测的下一时刻质心
        predicted_next_positions = current_positions + predicted_displacement
        predicted_next_centroid = torch.mean(predicted_next_positions, dim=0)  # (2,)
        
        # 计算真实的下一时刻质心
        target_next_centroid = torch.mean(target_positions, dim=0)  # (2,)
        
        # 质心偏移损失：预测质心应该与当前质心相同（守恒）
        centroid_loss = torch.mean((predicted_next_centroid - current_centroid) ** 2)
        
        # 总损失
        loss = mse_loss + 0 * centroid_loss
        # 反向传播和优化
        optimizer.zero_grad()
        loss.backward()

        # --------------------------
        # 梯度裁剪（在 backward 之后，step 之前）
        if max_grad_norm is not None and max_grad_norm > 0:
            params_with_grad = [p for group in optimizer.param_groups for p in group['params'] if p.grad is not None]
            if len(params_with_grad) > 0:
                torch.nn.utils.clip_grad_norm_(params_with_grad, max_grad_norm)
        if max_grad_value is not None:
            params_with_grad = [p for group in optimizer.param_groups for p in group['params'] if p.grad is not None]
            if len(params_with_grad) > 0:
                torch.nn.utils.clip_grad_value_(params_with_grad, max_grad_value)
        # --------------------------

        optimizer.step()

        # 累积损失用于计算epoch平均值
        epoch_loss += loss.item()
        batch_count += 1

    avg_loss = epoch_loss / batch_count if batch_count > 0 else 0.0
    
    # 定期保存模型检查点
    if current_epoch % save_epoch == 0 and current_epoch > 0:
        check_point_name=f'epoch={current_epoch}.pth'
        check_point_dir=os.path.join(save_dir,check_point_name)
        model.save_checkpoint(check_point_dir)

    # 返回epoch平均损失
    return avg_loss

In [ ]:
if __name__ == '__main__':
    print('=' * 60)
    print('Cell Motion Prediction Training')
    print('=' * 60)
    print(f'Sequence length: {sequence_length}')
    print(f'Node input size: {node_input_size}')
    print(f'Edge input size: {edge_input_size}')
    print(f'Batch size: {batch_size}')
    print(f'Max epochs: {max_epochs}')
    print('=' * 60)
    
    # 加载细胞运动数据集
    dataset_cell = CellMotion(
        dataset_dir=dataset_dir,
        split='train',
        max_epochs=max_epochs,
        sequence_length=sequence_length
    )
    
    # 创建数据加载器
    train_loader = DataLoader(
        dataset=dataset_cell,
        batch_size=batch_size,
        num_workers=0  # 单进程加载
    )
    
    # 定义图变换流程
    # 对于细胞运动，我们使用边索引而不是面
    # 1.  Cartesian: 添加笛卡尔坐标信息（不归一化）
    # 2. Distance: 添加边的距离信息（不归一化）
    transformer = T.Compose([
        T. Cartesian(norm=False),
        T.Distance(norm=False)
    ])

    # 创建训练日志文件
    # 使用时间戳作为文件名，避免覆盖之前的训练记录
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    log_dir = save_dir
    os.makedirs(log_dir, exist_ok=True)
    log_file = os.path.join(log_dir, f'training_loss_{timestamp}.csv')
    
    # 创建best_epoch文件夹用于保存最佳检查点
    best_epoch_dir = os.path.join(save_dir, 'best_epoch')
    os.makedirs(best_epoch_dir, exist_ok=True)
    
    print(f'\nTraining log will be saved to: {log_file}')
    print(f'Best epoch checkpoint will be saved to: {best_epoch_dir}\n')
    
    # 创建CSV文件并写入表头
    with open(log_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Epoch', 'Average_Loss', 'Timestamp', 'Is_Best'])
    
    # 初始化最佳loss和最佳epoch
    best_loss = float('inf')
    best_epoch_num = 0
    
    # 开始训练
    print('\nStarting training...\n')
    try:
        # 注意：CellMotion数据集是IterableDataset，会自动处理epoch
        # 这里的epoch是由数据集内部控制的
        epoch = 0
        for epoch in range(max_epochs):  
            avg_loss = train(simulator, train_loader, optimizer, epoch+1)
            
            # 判断是否是最佳检查点
            is_best = avg_loss < best_loss
            if is_best:
                best_loss = avg_loss
                best_epoch_num = epoch + 1
                print(f'\nNew best loss: {best_loss:.4e} (Previous best: {best_loss:.4e})')
                
                # 保存最佳检查点到best_epoch文件夹
                best_checkpoint_path = os.path.join(
                    best_epoch_dir, 
                    f'best_checkpoint_epoch.pth'
                )
                simulator.save_checkpoint(best_checkpoint_path)
            
            print(f'Epoch {epoch+1} completed | Average Loss: {avg_loss:.4e}')

            # 将loss记录到CSV文件
            with open(log_file, 'a', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                current_time = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
                writer. writerow([epoch+1, avg_loss, current_time])
            
            
    except KeyboardInterrupt:
        print('\n\nTraining interrupted by user')
        simulator.save_checkpoint()
        print('Final checkpoint saved')
    
    print('\nTraining completed!')
    print(f'Training log saved to: {log_file}')
    print(f'Best epoch: {best_epoch_num} with loss: {best_loss:.4e}')
    print(f'Best checkpoint location: {best_epoch_dir}')

Cell Motion Prediction Training
Sequence length: 20
Node input size: 41
Edge input size: 3
Batch size: 32
Max epochs: 1500
CellMotion Dataset /data/home/zhaoyj/wansheng/meshgn/meshGraphNets/dataset/train.h5 Initialized (sequence_length=20)

Training log will be saved to: checkpoint/20251222_143532/training_loss_20251222_143535.csv
Best epoch checkpoint will be saved to: checkpoint/20251222_143532/best_epoch


Starting training...


New best loss: 9.5969e-01 (Previous best: 9.5969e-01)
Simulator model saved at checkpoint/20251222_143532/best_epoch/best_checkpoint_epoch.pth
Epoch 1 completed | Average Loss: 9.5969e-01

New best loss: 7.6036e-01 (Previous best: 7.6036e-01)
Simulator model saved at checkpoint/20251222_143532/best_epoch/best_checkpoint_epoch.pth
Epoch 2 completed | Average Loss: 7.6036e-01

New best loss: 5.7280e-01 (Previous best: 5.7280e-01)
Simulator model saved at checkpoint/20251222_143532/best_epoch/best_checkpoint_epoch.pth
Epoch 3 completed | Average Loss: 5.7280e-0